# KV Retrieve Algorithm Discovery

This notebook is not another tracking notebook.

It is the first pass at a dedicated algorithm-discovery stack:

1. define symbolic task variables
2. generate controlled prompt sweeps
3. record candidate internal states
4. score variable recovery at each site
5. extract operator-level truth tables for `L2H1` and `L2H0`
6. run class-conditional replacement on the upstream `L1H0` state
7. score simple latent programs
8. report whether checkpoint-series origin analysis is even possible from current artifacts

The goal is to move from prompt traces toward reusable latent variables and operators.


## Setup

This cell loads the toy model, dataset, and all new algorithm-extraction helpers.

Why this matters:

- the notebook reuses the same live model/cache code as the existing KV analysis stack
- the new experiment flow is explicit about scope and artifacts
- if a required file is missing, the notebook fails immediately rather than hiding the error


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import torch

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

if Path.cwd().name == "notebook":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.kv_algorithm_oracle import (
    annotate_row,
    build_position_annotation_table,
    build_prompt_annotation_table,
)
from scripts.kv_algorithm_sweeps import (
    build_sweep_summary_table,
    generate_controlled_sweeps,
)
from scripts.kv_algorithm_record import (
    SITE_REGISTRY,
    build_recording_summary_table,
    build_site_dataset,
    record_prompt_rows,
)
from scripts.kv_algorithm_variable_finder import (
    build_family_variable_stability_summary_table,
    build_family_variable_stability_table,
    build_site_variable_ranking_table,
    build_variable_recovery_table,
)
from scripts.kv_algorithm_operator_finder import (
    build_head_attention_operator_summary_table,
    build_head_attention_operator_table,
    build_head_attention_family_stability_table,
    build_l2h0_copy_rule_summary_table,
    build_l2h0_copy_rule_table,
    build_l2h0_copy_rule_family_stability_table,
)
from scripts.kv_algorithm_causal_judge import (
    build_family_query_replacement_summary_table,
    build_family_query_replacement_table,
)
from scripts.kv_algorithm_program_score import (
    build_program_family_score_table,
    build_program_prediction_table,
    build_program_score_table,
)
from scripts.kv_algorithm_checkpoint_tracker import (
    build_checkpoint_availability_table,
    build_checkpoint_metadata_table,
    discover_checkpoint_series,
    summarize_checkpoint_tracker,
)
from scripts.kv_retrieve_analysis import (
    load_checkpoint_model,
    load_dataset_bundle,
    run_prompt,
)

DEVICE = torch.device("cpu")
DATASET_DIR = PROJECT_ROOT / "dataset" / "kv_retrieve_3"
CHECKPOINT_PATH = PROJECT_ROOT / "models" / "kv_retrieve_3" / "selected_checkpoint.pt"
MODEL_DIR = PROJECT_ROOT / "models" / "kv_retrieve_3"

for required_path in [DATASET_DIR, CHECKPOINT_PATH, MODEL_DIR]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required artifact: {required_path}")

bundle = load_dataset_bundle(DATASET_DIR)
checkpoint, analysis_model = load_checkpoint_model(CHECKPOINT_PATH, device=DEVICE)

TRAIN_RECORD_LIMIT = 128
BASE_SWEEP_ROWS = 24

train_rows = bundle.raw_splits["train"][:TRAIN_RECORD_LIMIT]
base_rows = bundle.raw_splits["test"][:BASE_SWEEP_ROWS]
sweep_rows = generate_controlled_sweeps(base_rows)

candidate_sites = [
    "block1_final_resid",
    "block1_final_l1h0",
    "block1_final_mlp_out",
    "l2h0_final_q",
    "l2h0_final_out",
    "l2h1_final_out",
    "final_hidden",
]


## Symbolic Oracle

Before asking what the model computes, we need exact task variables.

This section turns prompts into explicit labels such as:

- `query_key`
- `matching_slot`
- `selected_value`
- per-position token roles

This is the difference between "interesting activation" and "testable hypothesis."


In [2]:
dataset_annotation_table = build_prompt_annotation_table(base_rows[:3])
position_annotation_table = build_position_annotation_table(base_rows[:2])

dataset_annotation_table[[
    "prompt_id",
    "query_key",
    "matching_slot",
    "selected_value",
    "slot_keys",
    "slot_values",
    "prompt",
]]


,prompt_id,query_key,matching_slot,selected_value,slot_keys,slot_values,prompt
0,test_000000,K4,1,V0,"[K7, K4, K6]","[V2, V0, V7]",<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K4 ->
1,test_000001,K7,2,V6,"[K1, K4, K7]","[V2, V3, V6]",<bos> K1 V2 ; K4 V3 ; K7 V6 ; Q K7 ->
2,test_000002,K6,0,V1,"[K6, K3, K5]","[V1, V0, V6]",<bos> K6 V1 ; K3 V0 ; K5 V6 ; Q K6 ->


## Controlled Sweeps

These sweeps vary one factor at a time:

- query key
- slot order
- value assignment

The point is to create prompt families where we can test invariances instead of relying on one anchor prompt.


In [3]:
sweep_summary_table = build_sweep_summary_table(sweep_rows)
sweep_summary_table.head(18)


,prompt_id,base_prompt_id,family_name,family_value,query_key,matching_slot,selected_value,slot_keys,slot_values,prompt
0,test_000000::query_key_0_K7,test_000000,query_key_sweep,K7,K7,0,V2,K7 | K4 | K6,V2 | V0 | V7,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K7 ->
1,test_000000::query_key_1_K4,test_000000,query_key_sweep,K4,K4,1,V0,K7 | K4 | K6,V2 | V0 | V7,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K4 ->
2,test_000000::query_key_2_K6,test_000000,query_key_sweep,K6,K6,2,V7,K7 | K4 | K6,V2 | V0 | V7,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K6 ->
3,test_000000::slot_perm_012,test_000000,slot_permutation,0-1-2,K4,1,V0,K7 | K4 | K6,V2 | V0 | V7,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K4 ->
4,test_000000::slot_perm_021,test_000000,slot_permutation,0-2-1,K4,2,V0,K7 | K6 | K4,V2 | V7 | V0,<bos> K7 V2 ; K6 V7 ; K4 V0 ; Q K4 ->
5,test_000000::slot_perm_102,test_000000,slot_permutation,1-0-2,K4,0,V0,K4 | K7 | K6,V0 | V2 | V7,<bos> K4 V0 ; K7 V2 ; K6 V7 ; Q K4 ->
6,test_000000::slot_perm_120,test_000000,slot_permutation,1-2-0,K4,0,V0,K4 | K6 | K7,V0 | V7 | V2,<bos> K4 V0 ; K6 V7 ; K7 V2 ; Q K4 ->
7,test_000000::slot_perm_201,test_000000,slot_permutation,2-0-1,K4,2,V0,K6 | K7 | K4,V7 | V2 | V0,<bos> K6 V7 ; K7 V2 ; K4 V0 ; Q K4 ->
8,test_000000::slot_perm_210,test_000000,slot_permutation,2-1-0,K4,1,V0,K6 | K4 | K7,V7 | V0 | V2,<bos> K6 V7 ; K4 V0 ; K7 V2 ; Q K4 ->
9,test_000000::value_perm_012,test_000000,value_permutation,0-1-2,K4,1,V0,K7 | K4 | K6,V2 | V0 | V7,<bos> K7 V2 ; K4 V0 ; K6 V7 ; Q K4 ->


## Recorder

This section records a small train set and the controlled sweeps at a focused set of candidate sites.

The current candidate graph is intentionally narrow:

- `block1_final_l1h0`
- block-1 MLP output
- `L2H0.Q`
- `L2H0` output
- `L2H1` output
- final hidden state

The point is to reuse the current circuit knowledge rather than exploding the search space.


In [4]:
train_recorded = record_prompt_rows(analysis_model, bundle, train_rows, DEVICE)
sweep_recorded = record_prompt_rows(analysis_model, bundle, sweep_rows, DEVICE)

combined_recorded = train_recorded + sweep_recorded
recording_summary_table = build_recording_summary_table(combined_recorded)
recorded_site_dataset = build_site_dataset(analysis_model, combined_recorded, candidate_sites)

recording_summary_table.head(12)


,prompt_id,base_prompt_id,family_name,family_value,query_key,matching_slot,selected_value,predicted_token,correct,margin,prompt
0,train_000000,train_000000,dataset,original,K5,2,V0,V0,True,6.716457,<bos> K3 V4 ; K1 V7 ; K5 V0 ; Q K5 ->
1,train_000001,train_000001,dataset,original,K1,0,V3,V3,True,7.110188,<bos> K1 V3 ; K0 V0 ; K4 V7 ; Q K1 ->
2,train_000002,train_000002,dataset,original,K1,2,V0,V0,True,11.108434,<bos> K4 V6 ; K3 V4 ; K1 V0 ; Q K1 ->
3,train_000003,train_000003,dataset,original,K6,1,V1,V1,True,13.131505,<bos> K4 V0 ; K6 V1 ; K0 V6 ; Q K6 ->
4,train_000004,train_000004,dataset,original,K1,0,V4,V4,True,11.463393,<bos> K1 V4 ; K6 V1 ; K4 V2 ; Q K1 ->
5,train_000005,train_000005,dataset,original,K3,1,V1,V1,True,9.791921,<bos> K2 V4 ; K3 V1 ; K0 V0 ; Q K3 ->
6,train_000006,train_000006,dataset,original,K6,0,V7,V7,True,12.252010,<bos> K6 V7 ; K2 V3 ; K7 V4 ; Q K6 ->
7,train_000007,train_000007,dataset,original,K5,1,V4,V4,True,8.321630,<bos> K2 V1 ; K5 V4 ; K1 V2 ; Q K5 ->
8,train_000008,train_000008,dataset,original,K7,1,V1,V1,True,16.482112,<bos> K4 V4 ; K7 V1 ; K2 V0 ; Q K7 ->
9,train_000009,train_000009,dataset,original,K3,1,V5,V5,True,10.317435,<bos> K2 V0 ; K3 V5 ; K6 V7 ; Q K3 ->


## Variable Recovery

This is the first direct variable-discovery pass.

For each candidate site, we ask whether a simple linear readout can recover:

- `query_key`
- `matching_slot`
- `selected_value`

Training happens on a small dataset slice. Evaluation happens on the controlled sweeps.

A site is interesting only if the variable survives controlled variation.


In [5]:
train_mask = (
    (recorded_site_dataset.metadata["source_kind"] == "dataset")
    & (recorded_site_dataset.metadata["split"] == "train")
)
eval_mask = recorded_site_dataset.metadata["source_kind"] == "sweep"

variable_recovery_table = build_variable_recovery_table(
    recorded_site_dataset,
    sites=candidate_sites,
    variables=["query_key", "matching_slot", "selected_value"],
    train_mask=train_mask,
    eval_mask=eval_mask,
)
variable_ranking_table = build_site_variable_ranking_table(variable_recovery_table)

variable_ranking_table


,site,variable,train_rows,eval_rows,num_classes,chance_accuracy,eval_accuracy,eval_margin_over_chance,weight_norm
0,l2h0_final_out,matching_slot,128,360,3,0.333333,0.455556,0.122222,10.679283
1,l2h1_final_out,matching_slot,128,360,3,0.333333,0.425000,0.091667,7.225564
2,final_hidden,matching_slot,128,360,3,0.333333,0.388889,0.055556,3.274980
3,l2h1_final_out,query_key,128,360,8,0.125000,0.908333,0.783333,12.029532
4,final_hidden,query_key,128,360,8,0.125000,0.908333,0.783333,2.148906
5,block1_final_resid,query_key,128,360,8,0.125000,0.891667,0.766667,3.493657
6,final_hidden,selected_value,128,360,8,0.125000,0.950000,0.825000,2.181382
7,l2h0_final_out,selected_value,128,360,8,0.125000,0.897222,0.772222,9.820750
8,l2h1_final_out,selected_value,128,360,8,0.125000,0.786111,0.661111,9.751735


## Multi-Query Variable Stability

Pooled accuracy can hide brittle families.

This section asks whether the best site for each variable stays good across many separate prompt families.

The key question is not only "can the variable be decoded?" but:

- does it stay decodable across many different query families?
- where does it fail?


In [6]:
best_query_site = variable_ranking_table.query("variable == 'query_key'").iloc[0]["site"]
best_slot_site = variable_ranking_table.query("variable == 'matching_slot'").iloc[0]["site"]
best_value_site = variable_ranking_table.query("variable == 'selected_value'").iloc[0]["site"]

query_family_variable_table = build_family_variable_stability_table(
    recorded_site_dataset,
    site=best_query_site,
    variable="query_key",
    train_mask=train_mask,
    eval_mask=eval_mask,
)
slot_family_variable_table = build_family_variable_stability_table(
    recorded_site_dataset,
    site=best_slot_site,
    variable="matching_slot",
    train_mask=train_mask,
    eval_mask=eval_mask,
)
value_family_variable_table = build_family_variable_stability_table(
    recorded_site_dataset,
    site=best_value_site,
    variable="selected_value",
    train_mask=train_mask,
    eval_mask=eval_mask,
)

family_variable_summary_table = pd.concat(
    [
        build_family_variable_stability_summary_table(query_family_variable_table),
        build_family_variable_stability_summary_table(slot_family_variable_table),
        build_family_variable_stability_summary_table(value_family_variable_table),
    ],
    ignore_index=True,
)

family_variable_summary_table


,site,variable,families,family_accuracy_mean,family_accuracy_min,family_accuracy_max
0,l2h1_final_out,query_key,24,0.908333,0.466667,1.0
1,l2h0_final_out,matching_slot,24,0.455556,0.200000,0.8
2,final_hidden,selected_value,24,0.950000,0.666667,1.0


## Operator Discovery

The current highest-value operators are:

- `L2H1` as a routing / slot-selection candidate
- `L2H0` as a read/write candidate

We build truth-table style summaries instead of more saliency tables.


In [7]:
l2h1_attention_table = build_head_attention_operator_table(
    sweep_recorded,
    layer_index=1,
    head_index=1,
)
l2h0_attention_table = build_head_attention_operator_table(
    sweep_recorded,
    layer_index=1,
    head_index=0,
)
l2h1_summary_table = build_head_attention_operator_summary_table(
    l2h1_attention_table,
    label="L2H1 routing candidate",
)
l2h0_summary_table = build_head_attention_operator_summary_table(
    l2h0_attention_table,
    label="L2H0 selection candidate",
)
l2h0_copy_rule_table = build_l2h0_copy_rule_table(
    analysis_model,
    bundle,
    sweep_recorded,
)
l2h0_copy_summary_table = build_l2h0_copy_rule_summary_table(l2h0_copy_rule_table)

pd.concat([l2h1_summary_table, l2h0_summary_table], ignore_index=True)


,label,rows,top_key_slot_matches,top_value_slot_matches,top_total_slot_matches,matching_key_attention_mean,matching_value_attention_mean,query_key_attention_mean
0,L2H1 routing candidate,360,0.383333,0.644444,0.586111,0.032727,0.184137,0.360099
1,L2H0 selection candidate,360,0.247222,0.752778,0.711111,0.008175,0.261325,0.105902


## Multi-Query Operator Stability

These family tables ask whether the same operator story stays stable across many base prompts.

Strong pooled averages are not enough.
We want to know if the same routing and copy behavior keeps showing up across many separate prompt families.


In [8]:
l2h1_family_stability_table = build_head_attention_family_stability_table(
    l2h1_attention_table,
    label="L2H1 routing candidate",
)
l2h0_family_stability_table = build_head_attention_family_stability_table(
    l2h0_attention_table,
    label="L2H0 selection candidate",
)
l2h0_copy_family_stability_table = build_l2h0_copy_rule_family_stability_table(
    l2h0_copy_rule_table,
)

pd.concat(
    [
        l2h1_family_stability_table.head(6),
        l2h0_family_stability_table.head(6),
    ],
    ignore_index=True,
)


,label,base_prompt_id,rows,top_key_slot_matches,top_value_slot_matches,top_total_slot_matches,matching_key_attention_mean,matching_value_attention_mean
0,L2H1 routing candidate,test_000007,15,0.600000,0.866667,0.866667,0.003256,0.489450
1,L2H1 routing candidate,test_000020,15,0.133333,0.933333,0.866667,0.000963,0.092998
2,L2H1 routing candidate,test_000006,15,0.800000,0.333333,0.800000,0.397934,0.164810
3,L2H1 routing candidate,test_000013,15,0.800000,0.600000,0.800000,0.080849,0.000361
4,L2H1 routing candidate,test_000011,15,0.733333,0.800000,0.733333,0.022303,0.042583
5,L2H1 routing candidate,test_000014,15,0.000000,0.800000,0.733333,0.000040,0.534697
6,L2H0 selection candidate,test_000003,15,0.666667,1.000000,1.000000,0.033335,0.194885
7,L2H0 selection candidate,test_000004,15,0.533333,0.933333,0.933333,0.014694,0.140247
8,L2H0 selection candidate,test_000007,15,0.666667,0.933333,0.933333,0.001727,0.180573
9,L2H0 selection candidate,test_000014,15,0.066667,0.933333,0.933333,0.008654,0.160329


## Causal Judge: Within-Family Query Replacement

This is the multi-query version of the upstream replacement test.

For each base prompt family, we use the family's own query-key sweep rows as replacement sources.

That means:

- same context
- different query-conditioned upstream state
- test whether replacing the `L1H0` final-position state changes downstream slot choice and answer the way the query class predicts


In [9]:
query_sweep_recorded = [
    recorded
    for recorded in sweep_recorded
    if recorded.annotation.family_name == "query_key_sweep"
]
family_replacement_tables = []
for base_prompt_id in sorted({recorded.annotation.base_prompt_id for recorded in query_sweep_recorded}):
    family_prompt_group = [
        recorded
        for recorded in query_sweep_recorded
        if recorded.annotation.base_prompt_id == base_prompt_id
    ]
    family_source_patch = {
        "layer_index": 0,
        "kind": "head_resid",
        "head_index": 0,
        "source_positions": [family_prompt_group[0].annotation.arrow_position],
    }
    family_replacement_tables.append(
        build_family_query_replacement_table(
            analysis_model,
            bundle,
            family_prompt_group,
            source_patch=family_source_patch,
            destination_layer_index=1,
            device=DEVICE,
            analysis_head_index=0,
        )
    )

family_query_replacement_table = pd.concat(family_replacement_tables, ignore_index=True)
family_query_replacement_summary_table = build_family_query_replacement_summary_table(
    family_query_replacement_table
)

family_query_replacement_summary_table.head(12)


,base_prompt_id,rows,answer_switch_accuracy,slot_switch_accuracy,margin_mean
0,test_000002,9,1.000000,0.666667,10.464214
1,test_000003,9,1.000000,0.888889,7.685466
2,test_000007,9,1.000000,0.666667,12.082460
3,test_000013,9,1.000000,0.777778,12.650295
4,test_000021,9,1.000000,0.888889,10.623779
5,test_000000,9,0.888889,0.888889,12.303974
6,test_000004,9,0.888889,0.555556,9.637258
7,test_000006,9,0.888889,0.555556,9.998729
8,test_000010,9,0.888889,1.000000,9.315451
9,test_000011,9,0.777778,0.777778,7.615541


## Program Scoring

A program is scored by:

- slot prediction accuracy
- selected value prediction accuracy

The first baseline programs are intentionally simple:

- use `L2H0` attention to choose a slot, then copy that slot's value
- use `L2H1` attention to route to a slot, then copy that slot's value


In [10]:
sweep_site_dataset = build_site_dataset(analysis_model, sweep_recorded, candidate_sites)
program_prediction_table = build_program_prediction_table(
    sweep_site_dataset.metadata,
    l2h1_attention_table,
    l2h0_attention_table,
)
program_score_table = build_program_score_table(program_prediction_table)

program_score_table


,strategy,rows,operator_count,slot_accuracy,value_accuracy
0,l2h0_value_copy,360,1,0.752778,0.752778
1,l2h0_total_attention_copy,360,1,0.711111,0.711111
2,l2h1_total_route_then_copy,360,2,0.586111,0.586111
3,l2h1_key_route_then_copy,360,2,0.383333,0.383333


## Multi-Query Program Stability

A pooled program score can still hide brittle prompt families.

This section checks whether the same simple latent programs stay accurate across many base prompts rather than only on average.


In [11]:
program_family_score_table = build_program_family_score_table(program_prediction_table)
program_family_summary_table = (
    program_family_score_table
    .groupby("strategy", as_index=False)
    .agg(
        families=("base_prompt_id", "nunique"),
        value_accuracy_mean=("value_accuracy", "mean"),
        value_accuracy_min=("value_accuracy", "min"),
        value_accuracy_max=("value_accuracy", "max"),
        slot_accuracy_mean=("slot_accuracy", "mean"),
    )
    .sort_values(["value_accuracy_mean", "slot_accuracy_mean"], ascending=[False, False])
    .reset_index(drop=True)
)

program_family_summary_table


,strategy,families,value_accuracy_mean,value_accuracy_min,value_accuracy_max,slot_accuracy_mean
0,l2h0_value_copy,24,0.752778,0.200000,1.000000,0.752778
1,l2h0_total_attention_copy,24,0.711111,0.266667,1.000000,0.711111
2,l2h1_total_route_then_copy,24,0.586111,0.266667,0.866667,0.586111
3,l2h1_key_route_then_copy,24,0.383333,0.000000,0.800000,0.383333


## Checkpoint Tracker

If we want to study origin across training, we need a checkpoint series.

This section does not pretend that such a series exists if it does not. It reports the current artifact state directly.


In [12]:
checkpoint_availability_table = build_checkpoint_availability_table(MODEL_DIR)
checkpoint_summary_table = summarize_checkpoint_tracker(MODEL_DIR)
checkpoint_metadata_table = build_checkpoint_metadata_table(discover_checkpoint_series(MODEL_DIR))

checkpoint_summary_table


,checkpoint_count,has_checkpoint_series,message
0,1,False,"Only one non-SAE checkpoint is available, so training-time emergence cannot yet be evaluated."


## What This Notebook Now Gives You

This notebook adds missing meta-tools that the tracking notebook did not have:

- a symbolic oracle
- controlled sweep generation
- site-variable recovery scores
- multi-query family stability checks
- operator truth tables
- within-family query replacement
- simple latent-program scoring
- explicit checkpoint-series availability

It is still an early version.

What it does **not** claim yet:

- a fully solved internal algorithm
- a perfect variable basis
- training-origin analysis across a real checkpoint series

But it does create the right tool stack for moving from traces toward reusable learned operators.
